## Understanding BIDS
All EEG and iEEG data used in published studies by the Computational Memory Lab are also uploaded to OpenNeuro (https://openneuro.org/), a free and open platform for sharing MRI, PET, MEG, EEG, and iEEG data.

We will begin this tutorial by explaining the Brain Imaging Data Structure (BIDS) format and how to load BIDS data using open source tools (mne-bids). See more about the BIDS format here: https://bids.neuroimaging.io/index.html, https://bidsschematools.readthedocs.io/en/latest/doc_to_schema.html. 

### OpenBIDS Data Structure

The BIDS format provides consistent folder organization and data formatting structures. Below is the general folder structure we will be using for this course. More detail will be provided in later introductions.

<pre>
BIDS_Dataset_Collection/
├── PEERS/ (study root)
│    ├── sub-{subject_id}/
│    │   └── ses-{session_id}/
│    │       ├── beh/
│    │       │   ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_beh.json (Description of columns in the events data frame)
│    │       │   └── sub-{subject_id}_ses-{session_id}_task-{experiment}_beh.tsv  (The actual events file)
│    │       └── eeg/
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_eeg.bdf
│    │           └── sub-{subject_id}_ses-{session_id}_task-{experiment}_events.tsv
│    └── participants.tsv
└── Other Studies
</pre>

First, create a BIDS_Dataset_Collection folder which will store all study folders downloaded from OpenNeuro. Next, we can set the bids_root which indicates the root of the study folder.

In [1]:
# imports
import pandas as pd; pd.set_option('display.max_columns', None)
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
import numpy as np
import os
import sys
import mne
from ptsa.data.timeseries import TimeSeries

# Data comes from the cluster or OpenNeuro -- you are asked. See cml_data.py.
sys.path.insert(0, ".")
from cml_data import get_bids_root

We can use the get_entity_vals functon from the mne_bids package to find all the subjects and tasks in the database. Once we have selected a subject, we can navigate to the root of its folder and use the get_entity_vals function to find all sessions and tasks associated with that subject.

## Loading Scalp BIDS Data

To load data in BIDS, first specify which study root, subject, session, and task you desire and add them as to the BIDSPath object. After building the base BIDSPath, you can update the path with the datatype, suffix, and extension fields to specify which file (behavioral or eeg) you want. 

Here are the BIDSPath fields you can manipulate:
* root (str | Path: path to root of the study)
* subject (str: subject id)
* task (str: experiment id) (Scalp task examples: ltpFR, ltpFR2, VFFR)
* session (str: session id)
* datatype (str: type of data) (Examples: eeg (electrophysical), beh (behavioral))
* suffix (str: type of data) (Examples: beh, events, electrodes, coordsystem, channels)
* extension (str: file extension) (Examples: .tsv, .json, .edf, .bdf)

In [2]:
# plug in the subject, task, and session info into BIDSPath
# all inputs to BIDSPath should be strings so convert using the str() function

subject = "LTP093"
task = "ltpFR2"      # PEERS experiment 2 (scalp EEG)
session = 0

# Asks whether to use the cluster copy or download from OpenNeuro. This section
# loads the actual scalp recording further down, so we need the .edf too --
# about 740 MB for this session. It is cached, so you only download it once.
bids_root = get_bids_root(task, subject=subject, session=session,
                          include_timeseries=True)

base_path = BIDSPath(
                subject=subject,
                session=str(session),
                task=task,  
                root=bids_root
            )

ds004395 (ltpFR2): already downloaded -> bids_data/ds004395


### Behavioral Data
In the BIDS format, there are two locations where the behavioral data can be found. The first is stored in the beh (behavioral) folder. 

In [3]:
# specify the behavioral folder by setting the datatype and suffix to "beh" and setting extension to ".tsv"
beh_path = base_path.copy().update(
                datatype="beh", 
                suffix="beh",
                extension=".tsv",
            )

# load it using read_csv
evs_beh = pd.read_csv(beh_path.fpath, sep="\t")
evs_beh[:5]

,mstime,trial_type,stim_file,subject,experiment,session,trial,item_name,item_num,list,answer,test_x,test_y,test_z
0,0,SESS_START,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,82245,START,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN
2,82277,PROB,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,24.0,7.0,8.0,9.0
3,87951,PROB,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,20.0,3.0,8.0,9.0
4,90864,PROB,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,11.0,2.0,6.0,3.0


The second location is in the eeg folder.

In [4]:
# specify the eeg folder by setting the datatype to "eeg", suffix to "events", and setting extension to ".tsv"
eeg_path = base_path.copy().update(
                datatype="eeg", 
                suffix="events",
                extension=".tsv",
            )

# load it using read_csv
evs_eeg = pd.read_csv(eeg_path.fpath, sep="\t")
evs_eeg[:5]

,onset,duration,trial_type,sample,stim_file,subject,experiment,session,trial,item_name,item_num,list,answer,test_x,test_y,test_z
0,416.672,NaN,SESS_START,208336,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,498.918,NaN,START,249459,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN
2,498.950,NaN,PROB,249475,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,24.0,7.0,8.0,9.0
3,504.624,NaN,PROB,252312,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,20.0,3.0,8.0,9.0
4,507.536,NaN,PROB,253768,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,11.0,2.0,6.0,3.0


The only difference between the files is that the eeg/ieeg version describes time with the **onset** (start of event in seconds), **duration** (length of event in seconds), and **sample** (start of event in digital samples) variables and the behavioral version describes time with the **mstime** (start of event in milliseconds) variable. The differences are important for the packages to load EEG data as events, but you will not need to pay attention.

### EEG Data
To load EEG data, we first load the raw edf/bdf file and convert it into an mne.Epochs object.

In [5]:
# specify the datatype as "eeg", and say which file we want: the recording itself.
# Without suffix and extension, BIDSPath cannot tell the .edf from the sidecar .json
# files sitting next to it.
eeg_path = base_path.copy().update(datatype="eeg", suffix="eeg", extension=".edf")

# `on_ch_mismatch="warn"` is needed for this dataset. The channels.tsv sidecar calls
# the 129th electrode "Cz" while the recording itself calls it "E129" -- the same
# vertex reference channel under two names. Recent versions of mne-bids treat that
# disagreement as an error; older ones only warned. This is a good example of what
# working with real shared data is like: the science is fine, but small bookkeeping
# inconsistencies survive into published datasets and you have to decide what to do
# about them. Here it is safe to proceed.
raw = read_raw_bids(eeg_path, on_ch_mismatch="warn")
raw

Extracting EDF parameters from bids_data/ds004395/sub-LTP093/ses-0/eeg/sub-LTP093_ses-0_task-ltpFR2_eeg.edf...


Setting channel info structure...


Creating raw.info structure...


Reading channel info from bids_data/ds004395/sub-LTP093/ses-0/eeg/sub-LTP093_ses-0_task-ltpFR2_channels.tsv.


Reading electrode coords from bids_data/ds004395/sub-LTP093/ses-0/eeg/sub-LTP093_ses-0_space-CapTrak_electrodes.tsv.


Not fully anonymizing info - keeping hand, his_id, sex of subject_info


/var/folders/1w/wjpjylls40db25xym80gqn740000h_/T/ipykernel_10568/778250766.py:13: RuntimeWarning: Channel mismatch between bids_data/ds004395/sub-LTP093/ses-0/eeg/sub-LTP093_ses-0_task-ltpFR2_channels.tsv and the raw data file. Skipping channels.tsv-derived channel metadata.
  raw = read_raw_bids(eeg_path, on_ch_mismatch="warn")
/var/folders/1w/wjpjylls40db25xym80gqn740000h_/T/ipykernel_10568/778250766.py:13: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['E8', 'E25', 'E126', 'E127']
  raw = read_raw_bids(eeg_path, on_ch_mismatch="warn")
/var/folders/1w/wjpjylls40db25xym80gqn740000h_/T/ipykernel_10568/778250766.py:13: RuntimeWarning: DigMontage is only a subset of info. There is 1 channel position not present in the DigMontage. The channel missing from the montage is:

['E129'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if this is not an EEG channel, or use the on_missing parameter if the c

<RawEDF | sub-LTP093_ses-0_task-ltpFR2_eeg.edf, 129 x 2859000 (5718.0 s), ~148 KiB, data not loaded>

In [6]:
# constants
REL_START, REL_STOP = 200, 3000
BUFFER_MS = 1000
WIDTH = 6

FREQS = np.logspace(np.log10(2), np.log10(100), 46)
NOTCH_BAND = (58., 62.)
BATCH_EVENTS = 64

In [7]:
# get events from raw data's header
events, event_id = mne.events_from_annotations(raw)

# set min and max of Epochs window
tmin = (-BUFFER_MS / 1000)
tmax = ((REL_STOP /1000 + BUFFER_MS / 1000))

# load into MNE Epochs object
epochs_mne = mne.Epochs(
    raw,
    events=events,                      # we can now load the events here
    event_id=event_id,             # we can also load the filtered events here
    tmin=tmin,
    tmax=tmax,
    baseline=None,                     # must set to None to ignore automatic baselining
    preload=True,
    event_repeated="merge",
)

epochs_mne

Used Annotations descriptions: [np.str_('DISTRACTOR'), np.str_('PROB'), np.str_('REC_START'), np.str_('REC_WORD'), np.str_('REC_WORD_VV'), np.str_('REST_REWET'), np.str_('SESS_END'), np.str_('SESS_START'), np.str_('START'), np.str_('STOP'), np.str_('WORD')]


Multiple event values for single event times found. Creating new event value to reflect simultaneous events.


Not setting metadata


1328 matching events found


No baseline correction applied


0 projection items activated


Loading data for 1328 events and 2501 original time points ...


0 bad epochs dropped


<Epochs | 1328 events (all good), -1 – 4 s (baseline off), ~3.19 GiB, data loaded,
 np.str_('DISTRACTOR'): 23
 np.str_('PROB'): 392
 np.str_('REC_START'): 24
 np.str_('REC_WORD'): 232
 np.str_('REC_WORD_VV'): 2
 np.str_('REST_REWET'): 2
 np.str_('SESS_END'): 1
 np.str_('SESS_START'): 1
 np.str_('START'): 25
 np.str_('STOP'): 37
 and 3 more events ...>

## Loading RAM BIDS Data

The RAM dataset is the name for the CML intracranial eeg (ieeg) dataset, which is one of the largest ieeg datasets in the world. 

The folder structure for subjects with IEEGs implanted is different from the scalp. Scalp EEG contains a single edf file of monopolar electrodes and intracranial EEG contains two edf files for bipolar and monopolar electrodes and files describing the cooridnates, region, and other metadata for the electrodes.

<pre>
BIDS_Dataset_Collection/
├── PEERS/ (study root)
│    ├── sub-{subject_id}/
│    │   └── ses-{session_id}/
│    │       ├── beh/
│    │       │   ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_beh.json (Description of columns in the events data frame)
│    │       │   └── sub-{subject_id}_ses-{session_id}_task-{experiment}_beh.tsv  (The actual events file)
│    │       └── ieeg/
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-bipolar_channels.tsv           (Pair data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-bipolar_ieeg.edf               (Bipolar referenced signals)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-bipolar_ieeg.json               (Describes columns in pair data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-monopolar_channels.tsv         (Contact data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-monopolar_ieeg.edf             (Unreferenced signals)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-monopolar_ieeg.json            (Columns in contact data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_events.json                        (Description of columns in events file)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_events.tsv                         (Events file)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_space-{space_id}_coordsystem.json  (Indicates coordinate system and unit scale)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_space-{space_id}_electrodes.json   (Describes columns in electrodes data)
│    │           └── sub-{subject_id}_ses-{session_id}_task-{experiment}_space-{space_id}_electrodes.tsv    (Electrodes dataframe with full description)
│    └── participants.tsv
└── Other Studies
</pre>

The BIDSPath fields for RAM is the same as scalp, except with the additional acquisition field, which indicates whether the file has monopolar or bipolar data, and space, which lists hte iEEG coordinate system. Bipolar electrode data are differential, localized voltage measurements between pairs of electrodes, highlighting local neural activity rather than broad, reference-based potentials. 

Fields:
* root (str | Path: path to root of the study)
* subject (str: subject id)
* task (str: experiment id) (iEEG task examples: FR1, catFR1, PAL1, pyFR, RepFR1)
* session (str: session id)
* datatype (str: type of data) (Examples: ieeg (electrophysical), beh (behavioral))
* suffix (str: type of data) (Examples: beh, events, electrodes, channels, coordsystem)
* extension (str: file extension) (Examples: .tsv, .json, .edf, .bdf)
* acquisition (str: type of eeg acquisition ) (bipolar or monopolar)
* space (str: iEEG coordinate system) (MNI152NLin6ASym, Talarich) 

In [8]:

# plug in the subject, task, and session info into BIDSPath 
# all inputs to BIDSPath should be strings so convert using the str() function
subject = "R1111M"
task = "FR1"
session = 0

# This one needs the actual recordings. If you choose OpenNeuro you will be
# shown the size (~770 MB) and asked before anything downloads.
bids_root = get_bids_root(task, subject=subject, session=session,
                           include_timeseries=True)

base_path = BIDSPath(
    subject=subject,
    session=str(session),
    task=task,
    root=bids_root,
)


OpenNeuro ds004789 (FR1) — need 1 file(s), 437.0 MB to download.
  destination: bids_data/ds004789
  (this is the actual EEG recording; it is cached afterwards, so you only pay this once)
  downloading sub-R1111M_ses-0_task-FR1_acq-bipolar_ieeg.edf (437.0 MB)


      0.2%  1.0 MB / 437.0 MB

      0.5%  2.0 MB / 437.0 MB

      0.7%  3.0 MB / 437.0 MB

      0.9%  4.0 MB / 437.0 MB

...

     99.3%  434.0 MB / 437.0 MB

     99.5%  435.0 MB / 437.0 MB

     99.8%  436.0 MB / 437.0 MB

    100.0%  437.0 MB / 437.0 MB

    100.0%  437.0 MB / 437.0 MB

### Loading Monopolar EEG data

In [9]:
# specify the datatype as "ieeg" and acquisition to "monopolar"
acq = "monopolar"        
eeg_path = base_path.copy()
eeg_path.update(datatype="ieeg", acquisition="monopolar", suffix="ieeg", extension=".edf")

# load raw data
raw = read_raw_bids(eeg_path)

# constants
REL_START, REL_STOP = 200, 3000
BUFFER_MS = 1000
WIDTH = 6

FREQS = np.logspace(np.log10(2), np.log10(100), 46)
NOTCH_BAND = (58., 62.)
BATCH_EVENTS = 64

# get events from raw data's header
events, event_id = mne.events_from_annotations(raw)

# set min and max of Epochs window
tmin = (-BUFFER_MS / 1000)
tmax = ((REL_STOP /1000 + BUFFER_MS / 1000))

# load into MNE Epochs object
epochs_mne = mne.Epochs(
    raw,
    events=events,                      # we can now load the events here
    event_id=event_id,             # we can also load the filtered events here
    tmin=tmin,
    tmax=tmax,
    baseline=None, 
    preload=True,
    event_repeated="merge",
)

epochs_mne

Extracting EDF parameters from bids_data/ds004789/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_acq-monopolar_ieeg.edf...


Setting channel info structure...


Creating raw.info structure...


Reading channel info from bids_data/ds004789/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_acq-monopolar_channels.tsv.


Reading electrode coords from bids_data/ds004789/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_space-MNI152NLin6ASym_electrodes.tsv.


...

0 projection items activated


Loading data for 722 events and 2501 original time points ...


/var/folders/1w/wjpjylls40db25xym80gqn740000h_/T/ipykernel_10568/1310487713.py:7: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw = read_raw_bids(eeg_path)


1 bad epochs dropped


<Epochs | 721 events (all good), -1 – 4 s (baseline off), ~1.34 GiB, data loaded,
 np.str_('COUNTDOWN_END'): 25
 np.str_('COUNTDOWN_START'): 2
 np.str_('DISTRACT_END'): 24
 np.str_('DISTRACT_START'): 7
 np.str_('ORIENT'): 24
 np.str_('PRACTICE_DISTRACT_END'): 1
 np.str_('PRACTICE_REC_END'): 1
 np.str_('PRACTICE_REC_START'): 1
 np.str_('PRACTICE_WORD'): 12
 np.str_('PROB'): 82
 and 11 more events ...>

### Loading Bipolar EEG data

In [10]:
# specify the datatype as "ieeg" and acquisition to "bipolar"
eeg_path = base_path.copy()
eeg_path.update(datatype="ieeg", acquisition="bipolar", suffix="ieeg", extension=".edf")

# load raw data
raw = read_raw_bids(eeg_path)

# constants
REL_START, REL_STOP = 200, 3000
BUFFER_MS = 1000
WIDTH = 6

FREQS = np.logspace(np.log10(2), np.log10(100), 46)
NOTCH_BAND = (58., 62.)
BATCH_EVENTS = 64

# get events from raw data's header
events, event_id = mne.events_from_annotations(raw)

# set min and max of Epochs window
tmin = (-BUFFER_MS / 1000)
tmax = ((REL_STOP /1000 + BUFFER_MS / 1000))

# load into MNE Epochs object
epochs_mne = mne.Epochs(
    raw,
    events=events,                      # we can now load the events here
    event_id=event_id,             # we can also load the filtered events here
    tmin=tmin,
    tmax=tmax,
    baseline=None, 
    preload=True,
    event_repeated="merge",
)

epochs_mne

Extracting EDF parameters from bids_data/ds004789/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_acq-bipolar_ieeg.edf...


Setting channel info structure...


Creating raw.info structure...


Reading channel info from bids_data/ds004789/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_acq-bipolar_channels.tsv.


Reading electrode coords from bids_data/ds004789/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_space-MNI152NLin6ASym_electrodes.tsv.


...

0 projection items activated


Loading data for 722 events and 2501 original time points ...


/var/folders/1w/wjpjylls40db25xym80gqn740000h_/T/ipykernel_10568/260424670.py:6: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw = read_raw_bids(eeg_path)
/var/folders/1w/wjpjylls40db25xym80gqn740000h_/T/ipykernel_10568/260424670.py:6: RuntimeWarning: DigMontage is only a subset of info. There are 141 channel positions not present in the DigMontage. The channels missing from the montage are:

['LPOG1-LPOG9', 'LPOG1-LPOG2', 'LPOG2-LPOG10', 'LPOG2-LPOG3', 'LPOG3-LPOG4', 'LPOG3-LPOG11', 'LPOG4-LPOG5', 'LPOG4-LPOG12', 'LPOG5-LPOG6', 'LPOG5-LPOG13', 'LPOG6-LPOG7', 'LPOG6-LPOG14', 'LPOG7-LPOG8', 'LPOG7-LPOG15', 'LPOG8-LPOG16', 'LPOG9-LPOG17', 'LPOG9-LPOG10', 'LPOG10-LPOG11', 'LPOG10-LPOG18', 'LPOG11-LPOG12', 'LPOG11-LPOG19', 'LPOG12-LPOG13', 'LPOG12-LPOG20', 'LPOG13-LPOG14', 'LPOG13-LPOG21', 'LPOG14-LPOG15', 'LPOG14-LPOG22', 'LPOG15-LPOG16', 'LPOG15-LPOG23', 'LPOG16-LPOG24', 'LPOG17-LPOG25', 'LPOG17-LPOG18', 'LPOG18-LPOG19', 'LPOG18-LPOG26', 'LPOG19-LPOG20', 'LPOG

1 bad epochs dropped


<Epochs | 721 events (all good), -1 – 4 s (baseline off), ~1.89 GiB, data loaded,
 np.str_('COUNTDOWN_END'): 25
 np.str_('COUNTDOWN_START'): 2
 np.str_('DISTRACT_END'): 24
 np.str_('DISTRACT_START'): 7
 np.str_('ORIENT'): 24
 np.str_('PRACTICE_DISTRACT_END'): 1
 np.str_('PRACTICE_REC_END'): 1
 np.str_('PRACTICE_REC_START'): 1
 np.str_('PRACTICE_WORD'): 12
 np.str_('PROB'): 82
 and 11 more events ...>

### Loading Electrodes, Coordsystem, and Channel data

In [11]:
# electrodes 
eeg_path = base_path.copy()
eeg_path.update(datatype="ieeg", suffix="electrodes", space="MNI152NLin6ASym", extension=".tsv")
pd.read_csv(eeg_path.fpath, sep="\t")

,name,x,y,z,size,group,hemisphere,type,tal.x,tal.y,tal.z,wb.region,ind.region,stein.region
0,LPOG1,-67.9554,-20.436300,-26.318920,-999,LPOG,L,grid,-66.7592,-20.37470,-21.06940,NaN,middletemporal,NaN
1,LPOG2,-71.3723,-19.887300,-17.033223,-999,LPOG,L,grid,-68.5270,-19.30560,-13.11050,NaN,middletemporal,NaN
2,LPOG3,-69.4694,-16.873900,-5.837007,-999,LPOG,L,grid,-67.0028,-17.99460,-3.26183,NaN,middletemporal,NaN
3,LPOG4,-68.4177,-13.619100,6.599195,-999,LPOG,L,grid,-62.8222,-17.49650,6.35027,NaN,superiortemporal,NaN
4,LPOG5,-68.5695,-14.288000,16.220750,-999,LPOG,L,grid,-60.2654,-16.09750,16.38480,NaN,postcentral,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,LPS4,-60.3150,0.724204,-33.789406,-999,LPS,L,strip,-59.5567,-3.04176,-28.33260,NaN,middletemporal,NaN
96,LTD1,-24.4314,-20.139100,-23.616084,-999,LTD,L,depth,-23.8723,-23.65220,-19.49340,Left PHG parahippocampal gyrus,parahippocampal,Left EC
97,LTD2,-28.6866,-18.553800,-24.592941,-999,LTD,L,depth,-28.2112,-20.97920,-19.44620,Left Cerebral White Matter,parahippocampal,Left MTL WM
98,LTD3,-33.2518,-18.690900,-25.448912,-999,LTD,L,depth,-33.5565,-19.32430,-18.68380,Left PHG parahippocampal gyrus,parahippocampal,Left PRC


In [12]:
import json
# coordsystem 
eeg_path = base_path.copy()
eeg_path.update(datatype="ieeg", suffix="coordsystem", space="MNI152NLin6ASym", extension=".json")
with open(eeg_path.fpath, "r") as f:
    coordsystem = json.load(f)
coordsystem

{'iEEGCoordinateSystem': 'MNI152NLin6ASym', 'iEEGCoordinateUnits': 'mm'}

In [13]:
# monopolar channels 
eeg_path = base_path.copy()
eeg_path.update(datatype="ieeg", suffix="channels", acquisition="monopolar", extension=".tsv")
pd.read_csv(eeg_path.fpath, sep="\t")

,name,type,units,low_cutoff,high_cutoff,group,sampling_frequency,description,notch
0,LPOG1,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
1,LPOG2,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
2,LPOG3,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
3,LPOG4,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
4,LPOG5,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
...,...,...,...,...,...,...,...,...,...
95,LPS4,ECOG,V,NaN,NaN,LPS,500,strip,NaN
96,LTD1,SEEG,V,NaN,NaN,LTD,500,depth,NaN
97,LTD2,SEEG,V,NaN,NaN,LTD,500,depth,NaN
98,LTD3,SEEG,V,NaN,NaN,LTD,500,depth,NaN


In [14]:
# bipolar channels 
eeg_path = base_path.copy()
eeg_path.update(datatype="ieeg", suffix="channels", acquisition="bipolar", extension=".tsv")
pd.read_csv(eeg_path.fpath, sep="\t")

,name,type,units,low_cutoff,high_cutoff,reference,group,sampling_frequency,description,notch
0,LPOG1-LPOG9,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
1,LPOG1-LPOG2,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
2,LPOG2-LPOG10,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
3,LPOG2-LPOG3,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
4,LPOG3-LPOG4,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
...,...,...,...,...,...,...,...,...,...,...
136,LPS2-LPS3,ECOG,V,NaN,NaN,bipolar,LPS,500,strip,NaN
137,LPS3-LPS4,ECOG,V,NaN,NaN,bipolar,LPS,500,strip,NaN
138,LTD1-LTD2,SEEG,V,NaN,NaN,bipolar,LTD,500,depth,NaN
139,LTD2-LTD3,SEEG,V,NaN,NaN,bipolar,LTD,500,depth,NaN


If loading BIDS data seems complicated...it is! That's why we have created the **BIDSReader** package which simplifies loading BIDs data and replicates the syntax of CMLReader, the data reader package for the CML format. You will learn how to use **BIDSReader** in the next Introduction.